## working with agents on top of carteirinha extracted database

### this also should incorporate the base workflow style:
input: blob, id -> image, id -> llm ->  output: convenio, plano, nome da pessoa e número da carteirinha


In [67]:
import os
import base64
import sys
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Tuple
from PIL import Image   # noqa: F401
from pydantic import BaseModel as PydanticBaseModel
from pydantic import Field
from pydantic_settings import BaseSettings
import oracledb
import io
import cv2
import fitz
import numpy as np
import boto3
from botocore.config import Config
import json
import re
import time
from tqdm import tqdm
import mariadb
import pandas as pd
from collections import defaultdict
from dataclasses import dataclass, field
#strands agetinc workflow

#dont limit the visualization of all the colluns of a pandas dataframe
pd.set_option("display.max_columns", None)

# Add project root to Python path so we can import from app module
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)
# Now we can import from app (after adding to sys.path)
from app.utils.logger import get_logger
# Load environment variables from the project root directory
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

logger = get_logger(name=__name__)

## credentials config

In [118]:
@dataclass
class AppConstants:
    BEDROCK_DEFAULT_MODEL_ID: str = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
    BEDROCK_DEFAULT_MODEL_VERSION: str = "bedrock-2023-05-31"
    DEFAULT_PROMPTS_DIR: str = "prompts/"
    S3_BUCKET_NAME: str = "agente-ai-carteirinha"
    S3_RESULTS_PREFIX: str = "resultados"
    S3_DEBUG_PREFIX: str = "debug"
    STREAMING: bool = False
    CACHE_PROMPT = "default"
    RETRIES: Dict[str, int] = field(default_factory=lambda: {"max_attempts": 3, "mode": "standard"})
    CONNECTION_TIMEOUT: int = 5
    READ_TIMEOUT: int = 60
    TEMPERATURE: float = 1
    TOP_P: float = 0.95
    MAX_TOKENS: int = 4096
    BUDGET_TOKENS: int = 2048

In [69]:
class Settings(BaseSettings):
    """Carrega e valida as configurações a partir de variáveis de ambiente."""

    ORACLE_USER: str
    ORACLE_PASSWORD: str
    ORACLE_DSN: str
    ORACLE_INSTANT_CLIENT_PATH: Optional[str] = Field(
        None, alias="oracle_instant_client_path"
    )
    AWS_ACCESS_KEY_ID: str
    AWS_SECRET_ACCESS_KEY: str
    AWS_BEDROCK_REGION: str
    BEDROCK_MODEL_ID: str = AppConstants.BEDROCK_DEFAULT_MODEL_ID
    BEDROCK_MODEL_VERSION: str = AppConstants.BEDROCK_DEFAULT_MODEL_VERSION
    AWS_SERVICE_NAME: str
    MARIADB_USER: str
    MARIADB_PASSWORD: str
    MARIADB_HOST: str
    MARIADB_PORT: int = 3306
    MARIADB_DATABASE: str
    API_BASE_URL: Optional[str] = Field(None, alias="api_base_url")
    API_USERNAME: Optional[str] = Field(None, alias="username")
    API_PASSWORD: Optional[str] = Field(None, alias="password")

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"

In [70]:
def criar_boto3_client(
    service_name: str, settings: Settings, config: Optional[Config] = None
) -> boto3.client:
    try:
        logger.info(
            f"Criando cliente {service_name.upper()} para a região: {settings.AWS_BEDROCK_REGION}..."
        )
        client = boto3.client(
            service_name,
            region_name=settings.AWS_BEDROCK_REGION,
            aws_access_key_id=settings.AWS_ACCESS_KEY_ID,
            aws_secret_access_key=settings.AWS_SECRET_ACCESS_KEY,
            config=config,
        )
        logger.info(f"Cliente {service_name.upper()} criado com sucesso.")
        return client
    except Exception as e:
        logger.critical(f"Não foi possível criar o cliente {service_name.upper()}: {e}")
        raise


## geting the service ready to use
### model configuration

In [71]:
easy_prompt = "what are llm models?"

In [84]:
def safe_extract_json(raw_text: str) -> dict:
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw_text, re.DOTALL) \
        or re.search(r'\{.*?\}', raw_text, re.DOTALL)
    
    if not match:
        raise ValueError("Nenhum JSON encontrado na resposta do modelo.")

    json_str = match.group(1) if match.lastindex else match.group(0)
    json_str = json_str.strip()

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        logger.warning(f"JSON inválido detectado: {e}")
        # tenta corrigir
        cleaned = re.sub(r'^[^\{]*', '', json_str)
        cleaned = re.sub(r'[^\}]*$', '', cleaned)
        return json.loads(cleaned)

In [103]:
system_prompt = fr"""
Você é um especialista em extração estruturada de dados.  
Sua tarefa: analisar o texto e retornar **apenas** as informações de carteirinhas de convênio de saúde em JSON válido.
⚠️ Retorne **somente JSON**, sem explicações, comentários ou texto adicional.

Campos a extrair:
- "convenio": Nome do convênio
- "plano": Tipo do plano (ex.: "Plano Prata")
- "nome_pessoa": Nome completo do beneficiário
- "numero_carteirinha": Número da carteirinha processado conforme regras abaixo

Regras de limpeza:
1. Remover caracteres especiais usando regex `[^a-zA-Z0-9\s]`
2. Remover espaços no início e fim
3. Aplicar regras especiais por convênio antes de validar tamanho
4. Validar tamanho conforme lista de mapeamento
5. Se algum campo não puder ser identificado ou validado, retornar null

Regras especiais por convênio:
- CEMIG SAUDE: Se houver dois números, use a matrícula do beneficiário (não a matrícula antiga)
- SUL AMERICA: Se o número tiver mais de 17 dígitos, remover os 3 primeiros dígitos e manter os 17 últimos.
- Outros convênios: Validar tamanho conforme tabela; se não estiver na lista, retornar null

Tabela de mapeamento (convênio, número de dígitos esperado):
[
("STELLANTIS SAUDE MG", 17),
("SUL AMERICA", "variavel"),
("CASSI", 16),
("CAIXA ECONOMICA FEDERAL", 11),
("BLUE COMPANY", 16),
("POSTAL SAUDE - CORREIOS", 16),
("IPSM", 16),
("UNIMED SEGUROS", 16),
("BRADESCO", 15),
("BRADESCO OPERADORA", 15),
("PLAN ASSISTE - MPF", 14),
("CARE PLUS", 12),
("PETROBRAS - REGAP", 12),
("VALE - AMS", 12),
("FUNDAFFEMG", 12),
("CEMIG SAUDE", "variavel"),
("VALE - PASA", 10),
("AMIL", 9),
("AMIL VM (ANTIGA GOLDEN CROSS)", 9),
("COPASS", 8),
("SPA SAUDE", 5)
]

Exemplo:
Texto: "Paciente João da Silva, convênio SUL AMERICA, carteirinha 12345678901234567890"
JSON esperado:
{{
  "convenio": "SUL AMERICA",
  "plano": null,
  "nome_pessoa": "João da Silva",
  "numero_carteirinha": "45678901234567890"
}}

"""

In [128]:
# api format

from collections import defaultdict

class TextractKVExtractor:
    def __init__(self, textract_client):
        """
        textract_client: boto3 Textract client (already configured)
        """
        self.client = textract_client

    def _get_kv_map(self, file_bytes):
        response = self.client.analyze_document(
            Document={'Bytes': file_bytes},
            FeatureTypes=['FORMS']
        )
        blocks = response['Blocks']
        key_map, value_map, block_map = {}, {}, {}

        for block in blocks:
            block_map[block['Id']] = block
            if block['BlockType'] == 'KEY_VALUE_SET':
                if 'KEY' in block['EntityTypes']:
                    key_map[block['Id']] = block
                else:
                    value_map[block['Id']] = block
        return key_map, value_map, block_map

    def _find_value_block(self, key_block, value_map):
        for rel in key_block.get('Relationships', []):
            if rel['Type'] == 'VALUE':
                for value_id in rel['Ids']:
                    return value_map.get(value_id)
        return None

    def _get_text(self, block, block_map):
        if not block:
            return ''
        text = ''
        for rel in block.get('Relationships', []):
            if rel['Type'] == 'CHILD':
                for child_id in rel['Ids']:
                    child = block_map.get(child_id, {})
                    if child.get('BlockType') == 'WORD':
                        text += child.get('Text', '') + ' '
                    if child.get('BlockType') == 'SELECTION_ELEMENT' and child.get('SelectionStatus') == 'SELECTED':
                        text += 'X '
        return text.strip()

    def _get_kv_relationship(self, key_map, value_map, block_map):
        kvs = defaultdict(list)
        for key_id, key_block in key_map.items():
            value_block = self._find_value_block(key_block, value_map)
            key_text = self._get_text(key_block, block_map)
            value_text = self._get_text(value_block, block_map)
            kvs[key_text].append(value_text)
        return kvs

    def run(self, file_path=None, file_bytes=None):
        """
        Run the extraction.
        Provide either `file_path` or `file_bytes`.
        Returns a dict: {key: [values]}
        """
        if file_path:
            with open(file_path, 'rb') as f:
                file_bytes = f.read()
        if not file_bytes:
            raise ValueError("You must provide file_path or file_bytes.")

        key_map, value_map, block_map = self._get_kv_map(file_bytes)
        kvs = self._get_kv_relationship(key_map, value_map, block_map)
        return str(kvs)


In [129]:
# use text tract to extract keyword pairs

app_constants = AppConstants()
settings = Settings()

texttract_client = criar_boto3_client("textract", settings)
doc_path = "/home/joao/projects/company_projects/carteirinha-api/documents/pdf_carteirinha/LO_DOCUMENTO_ANEXO_CIRURGICO.pdf"

texttract_instance = TextractKVExtractor(texttract_client)

response = texttract_instance.run(file_path=doc_path)

{"timestamp": "2025-08-13T18:58:04", "level": "INFO", "name": "__main__", "message": "Criando cliente TEXTRACT para a região: us-east-1...", "filename": "788707453.py", "lineno": 5}
{"timestamp": "2025-08-13T18:58:04", "level": "INFO", "name": "__main__", "message": "Cliente TEXTRACT criado com sucesso.", "filename": "788707453.py", "lineno": 15}


In [105]:
# get the response to string:
response_str = str(response)

In [106]:


# Final payload
body = {
    "anthropic_version": app_constants.BEDROCK_DEFAULT_MODEL_VERSION,
    "max_tokens": 4096,
    "temperature": 1,
    "thinking": {
        "type": "enabled",
        "budget_tokens": 2048
    },
    "system": system_prompt,
    "messages": [
        {"role": "user", "content": [{"type": "text", "text": response_str}]},
    ]
}


In [107]:

app_constants = AppConstants()
settings = Settings()

# Create a custom boto3 session
bedrock_client = criar_boto3_client("bedrock-runtime", settings)

model_id = settings.BEDROCK_MODEL_ID
model_version = settings.BEDROCK_MODEL_VERSION

# Create a Bedrock model with the custom session


{"timestamp": "2025-08-13T18:19:43", "level": "INFO", "name": "__main__", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "788707453.py", "lineno": 5}
{"timestamp": "2025-08-13T18:19:43", "level": "INFO", "name": "__main__", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "788707453.py", "lineno": 15}


In [108]:
response = bedrock_client.invoke_model(
    body=json.dumps(body),
    modelId=model_id)

In [109]:
# Read raw response body
body_str = response["body"].read().decode("utf-8")  # decode if it's bytes
print("RAW BEDROCK BODY:", repr(body_str))
if not body_str.strip():
    raise ValueError("Empty body from Bedrock model")

# Parse JSON
response_body = json.loads(body_str)

# Extract text content
text_parts = [
    block.get("text", "")
    for block in response_body.get("content", [])
    if block.get("type") == "text"
]



RAW BEDROCK BODY: '{"id":"msg_bdrk_013tHZP3dpgCAfuUUcna1aJr","type":"message","role":"assistant","model":"claude-3-7-sonnet-20250219","content":[{"type":"thinking","thinking":"Vou analisar este texto para extrair as informações relevantes sobre a carteirinha de convênio.\\n\\n1. Primeiramente, identifico os campos relevantes:\\n   - Convenio: Parece ser \\"SulAmerica\\" (mencionado em \\"Elegibilidade Resultado SulAmerica\\")\\n   - Nome da pessoa: \\"LETICIA SCHNEIDER RIBEIRO\\"\\n   - Número da carteirinha: \\"557 88888 4834 0533 0026\\"\\n   - Plano: Vejo \\"Plano: 100\\", mas não sei se isso é o nome do plano ou apenas um código\\n\\n2. Aplicando as regras de limpeza para o número da carteirinha:\\n   - Removendo caracteres especiais e espaços: \\"55788888483405330026\\"\\n   - Como é SUL AMERICA e tem mais de 17 dígitos (tem 20 dígitos), devo remover os 3 primeiros dígitos\\n   - Então fica: \\"88888483405330026\\" (17 dígitos)\\n\\n3. Formatando o JSON final:\\n\\n```json\\n{\\n 

In [110]:
raw_text_response = "\n".join(text_parts)
parsed_json = safe_extract_json(raw_text_response)
parsed_json

{'convenio': 'SUL AMERICA',
 'plano': 'ADAPTADO',
 'nome_pessoa': 'LETICIA SCHNEIDER RIBEIRO',
 'numero_carteirinha': '88888483405330026'}

In [87]:
# save cleaned json to .json file
with open("output_test.json", "w") as json_file:
    json.dump(parsed_json, json_file, indent=4)


# creating a reusable way of data extraction

We want to input the image and get the parsed json as result

In [125]:
class AnthropicLLMService():
    def __init__(self, model_id: str, model_version: str, client: boto3.client, system_prompt: str, max_tokens: int, temperature: float, budget_tokens: int) -> None:
        self.model_id = model_id
        self.model_version = model_version
        self.client = client
        self.system_prompt = system_prompt
        self.MAX_TOKENS = max_tokens
        self.TEMPERATURE = temperature
        self.BUDGET_TOKENS = budget_tokens

    def _config_body(self, input_str: str) -> Dict[str, Any]:
        body = {
                "anthropic_version": self.model_version,
                "max_tokens": self.MAX_TOKENS,
                "temperature": self.TEMPERATURE,
                "thinking": {
                    "type": "enabled",
                    "budget_tokens": self.BUDGET_TOKENS
                },
                "system": self.system_prompt,
                "messages": [
                    {"role": "user", "content": [{"type": "text", "text": input_str}]},
                ]
            }
        return body

    def _safe_extract_json(raw_text: str) -> dict:
        match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw_text, re.DOTALL) \
            or re.search(r'\{.*?\}', raw_text, re.DOTALL)
        
        if not match:
            raise ValueError("Nenhum JSON encontrado na resposta do modelo.")

        json_str = match.group(1) if match.lastindex else match.group(0)
        json_str = json_str.strip()

        try:
            return json.loads(json_str)
        except json.JSONDecodeError as e:
            logger.warning(f"JSON inválido detectado: {e}")
            # tenta corrigir
            cleaned = re.sub(r'^[^\{]*', '', json_str)
            cleaned = re.sub(r'[^\}]*$', '', cleaned)
            return json.loads(cleaned)

    def invoke_model(self, input_str: str) -> dict:
        response = self.client.invoke_model(
            body=json.dumps(self._config_body(input_str)),
            modelId=self.model_id
        )
        body_str = response["body"].read().decode("utf-8")  # decode if it's bytes
        if not body_str.strip():
            raise ValueError("Empty body from Bedrock model")

        response_body = json.loads(body_str)

        # Extract text content
        text_parts = [
            block.get("text", "")
            for block in response_body.get("content", [])
            if block.get("type") == "text"
        ]

        raw_text_response = "\n".join(text_parts)
        parsed_json = safe_extract_json(raw_text_response)
        return parsed_json



In [130]:
## initialize services and variables
app_constants = AppConstants()
settings = Settings()


texttract_client = criar_boto3_client("textract", settings)
bedrock_client = criar_boto3_client("bedrock-runtime", settings)

texttract_instance = TextractKVExtractor(texttract_client)
llm_instance = AnthropicLLMService(
    model_id=settings.BEDROCK_MODEL_ID,
    model_version=settings.BEDROCK_MODEL_VERSION,
    client=bedrock_client,
    system_prompt=system_prompt,
    max_tokens=app_constants.MAX_TOKENS,
    temperature=app_constants.TEMPERATURE,
    budget_tokens=app_constants.BUDGET_TOKENS
)



{"timestamp": "2025-08-13T18:58:24", "level": "INFO", "name": "__main__", "message": "Criando cliente TEXTRACT para a região: us-east-1...", "filename": "788707453.py", "lineno": 5}
{"timestamp": "2025-08-13T18:58:24", "level": "INFO", "name": "__main__", "message": "Cliente TEXTRACT criado com sucesso.", "filename": "788707453.py", "lineno": 15}
{"timestamp": "2025-08-13T18:58:24", "level": "INFO", "name": "__main__", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "788707453.py", "lineno": 5}
{"timestamp": "2025-08-13T18:58:24", "level": "INFO", "name": "__main__", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "788707453.py", "lineno": 15}


In [131]:
# testing against data
path_file = "/home/joao/projects/company_projects/carteirinha-api/documents/pdf_carteirinha/LO_DOCUMENTO_ANEXO_CIRURGICO.pdf"
textract_text_response = texttract_instance.run(file_path=path_file)
llm_response = llm_instance.invoke_model(textract_text_response)
llm_response

{'convenio': 'SUL AMERICA',
 'plano': 'ADAPTADO',
 'nome_pessoa': 'LETICIA SCHNEIDER RIBEIRO',
 'numero_carteirinha': '88888483405330026'}